In [15]:
import pandas as pd
from typing import Any

- 비정형/지식 데이터 백터화하기 위한 데이터 처리 모듈 구성, 테스트

In [7]:
process_date = "2026-10-08"

def read_csv_s3(key:str) -> pd.DataFrame:
    # airflow에서는 s3_client 이용
    # 테스트 로컬에서 처리
    return pd.read_csv("../sample_data/silver/dt=2026-10-08/"+key)

In [19]:
def build_knowlegde_gold():
    # s3 버킷내 실버 경로
    base = f"silver/dt={process_date}"
    print( f"base => {base}")

    # 데이터 로드 
    # 상품 마스터
    products = read_csv_s3(f"products.csv")    
    # 고객 리뷰
    reviews = read_csv_s3(f"reviews.csv")
    # CS 문의
    cs_tickets = read_csv_s3(f"cs_tickets.csv")
    # 환불 기록
    refunds = read_csv_s3(f"refunds.csv")
    # 회사 정책 문서
    policies = read_csv_s3(f"policies.csv")
    print(products.shape, reviews.shape, cs_tickets.shape, refunds.shape, policies.shape)

    display(products)
    product_map = (
        products
        .set_index("product_id")["product_name"]
        .to_dict()
    )
    display(product_map)
    '''
        {
            'P001': 'Basic Desk',
            'P002': 'LED Desk Lamp',
            'P003': 'Storage Box',
            'P004': 'Cotton Bedding',
            'P007': 'Air Comfort Chair'
        }
    '''

    # 문서 구조가 서로 상이
    display(reviews.head(2))
    display(cs_tickets.head(2))
    display(refunds.head(2))
    display(policies.head(2))


    pass

build_knowlegde_gold()

base => silver/dt=2026-10-08
(5, 4) (254, 6) (70, 7) (23, 6) (2, 4)


,product_id,product_name,category,unit_price
0,P001,Basic Desk,office,120000
1,P002,LED Desk Lamp,office,39000
2,P003,Storage Box,living,29000
3,P004,Cotton Bedding,living,89000
4,P007,Air Comfort Chair,living,189000


{'P001': 'Basic Desk',
 'P002': 'LED Desk Lamp',
 'P003': 'Storage Box',
 'P004': 'Cotton Bedding',
 'P007': 'Air Comfort Chair'}

,review_id,order_id,product_id,rating,review_text,review_time
0,V000001,O000001,P004,5,배송 일정이 안내된 날짜와 거의 맞았고 포장 상태도 깔끔했습니다. 제품 설명과 실제...,2026-10-07T12:15:14
1,V000002,O000002,P001,5,제품 품질이 기대한 수준이었고 설치 과정도 어렵지 않았습니다. 주문부터 배송까지 특...,2026-10-07T11:32:38


,ticket_id,customer_id,order_id,product_id,topic,message,created_at
0,T000001,C00142,O000005,P002,product_question,상품 크기와 소재를 확인하고 싶습니다. 상세 페이지의 치수 외에 실제 사용 시 필요...,2026-10-06T22:40:39
1,T000002,C00164,O000008,P002,product_question,상품 크기와 소재를 확인하고 싶습니다. 상세 페이지의 치수 외에 실제 사용 시 필요...,2026-10-06T17:14:43


,refund_id,order_id,product_id,refund_reason,refund_amount,refund_time
0,R000001,O000038,P007,package_damage,567000,2026-10-07T16:53:17
1,R000002,O000063,P007,package_damage,378000,2026-10-07T06:39:34


,policy_id,policy_type,title,content
0,POL-DELIVERY-001,delivery,배송 지연 및 파손 처리 정책,배송 예정일을 초과한 주문은 배송사 조회 결과와 출고 시각을 확인한 뒤 고객에게 변...
1,POL-REFUND-001,refund,환불 및 반품 기본 정책,"상품 하자, 오배송, 배송 중 파손 등 판매자 또는 배송 과정에 책임이 있는 사유는..."


In [ ]:
def build_knowlegde_gold():
    # s3 버킷내 실버 경로
    base = f"silver/dt={process_date}"
    print( f"base => {base}")

    # 데이터 로드 
    # 상품 마스터
    products = read_csv_s3(f"products.csv")    
    # 고객 리뷰
    reviews = read_csv_s3(f"reviews.csv")
    # CS 문의
    cs_tickets = read_csv_s3(f"cs_tickets.csv")
    # 환불 기록
    refunds = read_csv_s3(f"refunds.csv")
    # 회사 정책 문서
    policies = read_csv_s3(f"policies.csv")

    # product_id로 제품명 dict 구성
    product_map = (
        products
        .set_index("product_id")["product_name"]
        .to_dict()
    )

    # 통합 스키마를 담는 그릇 준비
    docs: list[dict[str, Any]] = []

    # 리뷰 -> 지식 문서로 변환
    for row in reviews.to_dict("records"):
        docs.append({
            # 문서 ID -> 데이터를 구분하는 ID가 문서 ID로 대체됨
            "document_id"   : f"review:{row['review_id']}",  # V000001, V000002, ...
            # 처리일
            "process_date"  : process_date,
            # 문서 출처
            "source_type"   : "review",
            # 소스 ID -> 중복 사용
            "source_id"     : row['review_id'],
            # 어떤 제품에 대한 리뷰
            "product_id"    : row['product_id'],
            # 이벤트 시간
            "event_time"    : row['review_time'],
            # 타이틀
            "title"         : (
                f"{product_map.get(row['product_id'])} "
                "고객 리뷰"
            ),
            # 리뷰에 대한 핵심 내용 -> 자연형 문서 형태로 합침, 문장의 구분자로 .를 사용
            "content"       : (
                f"상품: {product_map.get(row['product_id'])}. "
                f"평점: {row['rating']}점. "
                f"고객 리뷰: {row['review_text']}"
            ),
            # 메타 데이터 -> 검색 결과와 함께 활용되는 구조화 정보
            "metadata"      : {
                "rating"   : int(row['rating']), 
                "order_id" : row['order_id']
            }
        })
    #print( docs )

    # CS -> 지식 문서로 변환
    for row in cs_tickets.to_dict("records"):
        docs.append({
            "document_id"   : f"cs:{row['ticket_id']}",  # V000001, V000002, ...
            "process_date"  : process_date,
            "source_type"   : "cs_ticket",
            "source_id"     : row['ticket_id'],
            "product_id"    : row['product_id'],
            "event_time"    : row['created_at'],
            "title"         : (
                f"{product_map.get(row['product_id'])} "
                "CS 문의"
            ),
            "content"       : (
                f"상품: {product_map.get(row['product_id'])}. "
                f"문의 우형: {row['topic']}. "
                f"고객 문의: {row['message']}"
            ),
            # 토픽, 고객 정보 메타 데이터 구성
            "metadata"      : {
                "topic"    : row['topic'], 
                "order_id" : row['order_id'],
                "customer_id" : row['customer_id']
            }
            
        })

        
    # 환불 -> 지식 문서로 변환
    for row in refunds.to_dict("records"):
        docs.append({
            "document_id"   : f"refund:{row['refund_id']}", 
            "process_date"  : process_date,
            "source_type"   : "refund",
            "source_id"     : row['refund_id'],
            "product_id"    : row['product_id'],
            "event_time"    : row['refund_time'],
            "title"         : (
                f"{product_map.get(row['product_id'])} "
                "환불 기록"
            ),
            "content"       : (
                f"상품: {product_map.get(row['product_id'])}. "
                f"문의 유형: {row['refund_reason']}. "
                f"고객 문의: {row['refund_amount']}원."
            ),
            # 환불 이유
            "metadata"      : {
                "refund_reason"    : row['refund_reason'], 
                "order_id" : row['order_id']
            }
        })
        
    # 정책 -> 지식 문서로 변환
    for row in policies.to_dict("records"):
        docs.append({
            "document_id"   : f"policy:{row['policy_id']}", 
            "process_date"  : process_date,
            "source_type"   : "policy",
            "source_id"     : row['policy_id'],
            "product_id"    : None,
            "event_time"    : None,
            "title"         : row['title'],
            "content"       : (
                f"정책명: {row['title']}. "
                f"정책 내용: {row['content']}"
            ),
            "metadata"      : {
                "policy_type"    : row['policy_type']
            }
        })
        
    print(docs)

build_knowlegde_gold()

base => silver/dt=2026-10-08
[{'document_id': 'review:V000001', 'process_date': '2026-10-08', 'source_type': 'review', 'source_id': 'V000001', 'product_id': 'P004', 'event_time': '2026-10-07T12:15:14', 'title': 'Cotton Bedding 고객 리뷰', 'content': '상품: Cotton Bedding. 평점: 5점. 고객 리뷰: 배송 일정이 안내된 날짜와 거의 맞았고 포장 상태도 깔끔했습니다. 제품 설명과 실제 상품의 차이가 크지 않아 전반적으로 만족합니다.', 'metadata': {'rating': 5, 'order_id': 'O000001'}}, {'document_id': 'review:V000002', 'process_date': '2026-10-08', 'source_type': 'review', 'source_id': 'V000002', 'product_id': 'P001', 'event_time': '2026-10-07T11:32:38', 'title': 'Basic Desk 고객 리뷰', 'content': '상품: Basic Desk. 평점: 5점. 고객 리뷰: 제품 품질이 기대한 수준이었고 설치 과정도 어렵지 않았습니다. 주문부터 배송까지 특별한 문제 없이 진행되어 재구매 의향이 있습니다.', 'metadata': {'rating': 5, 'order_id': 'O000002'}}, {'document_id': 'review:V000003', 'process_date': '2026-10-08', 'source_type': 'review', 'source_id': 'V000003', 'product_id': 'P004', 'event_time': '2026-10-07T19:27:21', 'title': 'Cotton Bedding 고객 리뷰', 'content': '상품:

In [21]:
policies

NameError: name 'policies' is not defined